In [ ]:
!git clone https://github.com/singularity-sans-null/NeuroScan.git

Cloning into 'NeuroScan'...
fatal: could not read Username for 'https://github.com': No such device or address


In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485,0.456,0.406],
        std = [0.229,0.224,0.225]
    )
])
train_path = "/content/drive/MyDrive/BrainTumorProject/Dataset/DATASET/classification/Training"

train_dataset = ImageFolder(
    root = train_path,
    transform = transform
)

train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = 32,
    shuffle = True
)


In [12]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
    )
# first we freeze the models features- no need to retrain them
for param in model.features.parameters():
  param.requires_grad = False

# update the predicition classes
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(1280, 4)
)

# Move the model to GPU

model = model.to(device)

In [ ]:
num_epochs= None
for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0
  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f"Epoch {epoch+1}: loss = {running_loss:.4f}")



In [ ]:
for images, labels in train_loader:
  optimizer.zero_grad()
  outputs = model(images)
  loss = criterion(outputs, labels)
  loss.backward()
  optimizer.step()
